# 울림 — 사이렌/경적 감지 모델 학습 (YAMNet 전이학습)

**결과물** (마지막 셀에서 `woollim_model.zip` 다운로드 → 저장소 `ml/` 폴더에 풀기)
- `woollim_sound.tflite` — 입력 `float32[15600]`(16 kHz 모노, -1..1, 0.975초) → 출력 `float32[2]` = `[siren, horn]` 확률
- `labels.txt`, `model_config.json`(임계값 포함), `metrics.json`
- `test_audio/` — 팀원이 Pi에서 `python inference_reference.py --check test_audio` 로 이식 확인

**실행**: 런타임 → 모두 실행. GPU 없어도 됩니다 (UrbanSound8K 포함 시 다운로드·임베딩 추출 30~50분).

**데이터**: ESC-50 (CC BY-NC), UrbanSound8K (CC BY-NC) — 비상업(학교 프로젝트) 용도.

In [ ]:
# 1. 설정
SR, WIN, HOP = 16000, 15600, 8000     # 배포 시 창/hop (YAMNet 1프레임 = 0.975 s)
YAM_HOP = 7680                         # YAMNet 내부 프레임 hop (0.48 s)
LABELS = ['siren', 'horn']
USE_US8K = True                        # False면 ESC-50만 (빠른 베이스라인)
AUG_COPIES = 2                         # 학습 클립당 '차 안' 증강본 개수
TARGET_RECALL = {'siren': 0.95, 'horn': 0.90}
SEED = 0

In [ ]:
# 2. 패키지 + YAMNet 코드/가중치 (tflite_compatible 모드로 만들어야 Pi용 TFLite 변환이 된다)
#    YAMNet 공식 코드는 Keras 2 방식이라, TF를 import 하기 전에 legacy Keras(tf-keras)를 켠다
import os, urllib.request
os.environ['TF_USE_LEGACY_KERAS'] = '1'
!pip -q install librosa soundfile
!pip -q install "tf-keras~=$(python -c 'import tensorflow as tf; print(tf.__version__.rsplit(".", 1)[0])').0"
os.makedirs('/content/yamnet', exist_ok=True)
base = 'https://raw.githubusercontent.com/tensorflow/models/master/research/audioset/yamnet/'
for f in ['yamnet.py', 'features.py', 'params.py', 'yamnet_class_map.csv']:
    urllib.request.urlretrieve(base + f, f'/content/yamnet/{f}')
if not os.path.exists('/content/yamnet/yamnet.h5'):
    urllib.request.urlretrieve('https://storage.googleapis.com/audioset/yamnet.h5', '/content/yamnet/yamnet.h5')

import sys; sys.path.insert(0, '/content/yamnet')
import numpy as np, pandas as pd, tensorflow as tf, librosa, soundfile as sf, scipy.signal, json
import params as yamnet_params, yamnet as yamnet_lib

P = yamnet_params.Params(tflite_compatible=True)
yam = yamnet_lib.yamnet_frames_model(P)
yam.load_weights('/content/yamnet/yamnet.h5')
print('TF', tf.__version__, '| YAMNet ready')

In [ ]:
# 3. 데이터 다운로드
DATA = '/content/data'; os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f'{DATA}/ESC-50-master'):
    !wget -q -O /content/esc50.zip https://github.com/karoldvl/ESC-50/archive/master.zip
    !unzip -q /content/esc50.zip -d {DATA}
if USE_US8K and not os.path.exists(f'{DATA}/UrbanSound8K'):
    !wget -q -O /content/us8k.tar.gz https://zenodo.org/records/1203745/files/UrbanSound8K.tar.gz
    !tar -xzf /content/us8k.tar.gz -C {DATA}

In [ ]:
# 4. 클립 목록 — split은 데이터셋 공식 fold 기준(같은 원본 녹음이 train/test에 섞이지 않음)
NOISE_CATS = {'engine', 'train', 'airplane', 'wind', 'rain', 'engine_idling', 'air_conditioner'}

esc = pd.read_csv(f'{DATA}/ESC-50-master/meta/esc50.csv')
esc = pd.DataFrame({
    'path': f'{DATA}/ESC-50-master/audio/' + esc.filename,
    'category': esc.category,
    'split': esc.fold.map(lambda f: 'test' if f == 5 else 'val' if f == 4 else 'train'),
    'group': 'fs' + esc.src_file.astype(str),
})
clips = [esc]
if USE_US8K:
    us = pd.read_csv(f'{DATA}/UrbanSound8K/metadata/UrbanSound8K.csv')
    us = us[~us.fsID.astype(str).isin(set(esc.group.str[2:]))]   # ESC-50과 같은 freesound 원본 제거
    clips.append(pd.DataFrame({
        'path': f'{DATA}/UrbanSound8K/audio/fold' + us.fold.astype(str) + '/' + us.slice_file_name,
        'category': us['class'],
        'split': us.fold.map(lambda f: 'test' if f == 10 else 'val' if f == 9 else 'train'),
        'group': 'fs' + us.fsID.astype(str),
    }))
clips = pd.concat(clips, ignore_index=True)
clips['siren'] = (clips.category == 'siren').astype(int)
clips['horn'] = (clips.category == 'car_horn').astype(int)
print(clips.groupby('split')[['siren', 'horn']].sum().assign(total=clips.groupby('split').size()))

In [ ]:
# 5. '차 안' 증강: 차체 감쇠(저역통과) + 엔진·도로 소음 믹스 + 게인 + 시간 이동
rng = np.random.default_rng(SEED)

def load(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    return y.astype(np.float32)

noise_rows = clips[(clips.split == 'train') & clips.category.isin(NOISE_CATS)]
noise_bank = [load(p) for p in noise_rows.sample(min(300, len(noise_rows)), random_state=SEED).path]
noise_bank = [n for n in noise_bank if np.sqrt(np.mean(n ** 2)) > 1e-4]
print('noise clips:', len(noise_bank))

def rms(x): return float(np.sqrt(np.mean(np.square(x, dtype=np.float64))) + 1e-10)

def augment(y, rng):
    """return (증강 신호, 같은 시간 정렬의 깨끗한 신호) — 깨끗한 신호는 프레임 라벨링(소리 구간 찾기)용"""
    shift = int(rng.integers(0, HOP))
    clean = np.concatenate([np.zeros(shift, np.float32), y])
    x = clean.copy()
    if rng.random() < 0.7:
        sos = scipy.signal.butter(4, rng.uniform(2000, 4000), 'low', fs=SR, output='sos')
        x = scipy.signal.sosfilt(sos, x).astype(np.float32)
    if rng.random() < 0.8:
        n = noise_bank[rng.integers(len(noise_bank))]
        n = np.tile(n, int(np.ceil(len(x) / len(n))))[:len(x)]
        snr = rng.uniform(-5, 20)
        x = x + n * (rms(x) / rms(n)) / (10 ** (snr / 20))
    x = x * 10 ** (rng.uniform(-10, 10) / 20)
    return np.clip(x, -1, 1).astype(np.float32), clean

In [ ]:
# 6. YAMNet 임베딩 추출 (프레임 단위) + 프레임 라벨
#    양성 클립: 가장 큰 프레임보다 12 dB 이상 작은 프레임(무음·잔향)은 애매하므로 버린다
def frames_db(y, n_frames):
    y = np.pad(y, (0, max(0, (n_frames - 1) * YAM_HOP + WIN - len(y))))
    return np.array([20 * np.log10(rms(y[i * YAM_HOP: i * YAM_HOP + WIN])) for i in range(n_frames)])

def embed(x):
    _, emb, _ = yam(x)
    return emb.numpy()

feats = {s: {'X': [], 'Y': [], 'G': [], 'C': [], 'K': []} for s in ['train', 'val', 'test_clean', 'test_car']}

def add(bucket, x, clean, row):
    emb = embed(x)
    y = np.array([row.siren, row.horn], np.float32)
    keep = np.ones(len(emb), bool)
    if y.any():
        db = frames_db(clean, len(emb))
        keep = db >= db.max() - 12
    b = feats[bucket]
    b['X'].append(emb[keep].astype(np.float16)); b['Y'].append(np.tile(y, (keep.sum(), 1)))
    n = int(keep.sum())
    b['G'] += [row.group] * n; b['C'] += [row.category] * n; b['K'] += [row.Index] * n

from tqdm.auto import tqdm
eval_rng = np.random.default_rng(SEED + 1)   # val/test 증강은 고정 시드
for row in tqdm(clips.itertuples(), total=len(clips)):
    y = load(row.path)
    if len(y) == 0: continue
    if row.split == 'train':
        add('train', y, y, row)
        for _ in range(AUG_COPIES):
            add('train', *augment(y, rng), row)
    elif row.split == 'val':
        add('val', y, y, row)
        add('val', *augment(y, eval_rng), row)
    else:
        add('test_clean', y, y, row)
        add('test_car', *augment(y, eval_rng), row)

D = {}
for k, b in feats.items():
    D[k] = dict(X=np.concatenate(b['X']).astype(np.float32), Y=np.concatenate(b['Y']),
                G=np.array(b['G']), C=np.array(b['C']), K=np.array(b['K']))
    print(f"{k:10s} frames={len(D[k]['X']):6d}  siren={int(D[k]['Y'][:, 0].sum()):5d}  horn={int(D[k]['Y'][:, 1].sum()):5d}")

In [ ]:
# 7. 분류 헤드 학습 (멀티라벨 sigmoid: 둘 다 낮으면 배경)
tf.keras.utils.set_random_seed(SEED)
Xtr, Ytr = D['train']['X'], D['train']['Y']
n_neg = (Ytr.sum(1) == 0).sum()
pos_w = {c: min(np.sqrt(n_neg / max(Ytr[:, i].sum(), 1)), 10.0) for i, c in enumerate(LABELS)}
w = np.ones(len(Ytr), np.float32)
for i, c in enumerate(LABELS):
    w = np.where(Ytr[:, i] == 1, np.maximum(w, pos_w[c]), w)
print('positive weights:', pos_w)

head = tf.keras.Sequential([
    tf.keras.layers.Input((1024,)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(LABELS), activation='sigmoid'),
], name='woollim_head')
head.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
             metrics=[tf.keras.metrics.AUC(curve='PR', multi_label=True, name='pr_auc')])
hist = head.fit(Xtr, Ytr, sample_weight=w, batch_size=256, epochs=60,
                validation_data=(D['val']['X'], D['val']['Y']),
                callbacks=[tf.keras.callbacks.EarlyStopping('val_pr_auc', mode='max', patience=6,
                                                            restore_best_weights=True)],
                verbose=2)

In [ ]:
# 8. 임계값 선택 (val): 목표 재현율을 만족하는 값 중 정밀도 최대
from sklearn.metrics import precision_recall_curve, average_precision_score, precision_score, recall_score

pv = head.predict(D['val']['X'], verbose=0)
THR = {}
for i, c in enumerate(LABELS):
    prec, rec, th = precision_recall_curve(D['val']['Y'][:, i], pv[:, i])
    ok = np.where(rec[:-1] >= TARGET_RECALL[c])[0]
    THR[c] = float(th[ok[np.argmax(prec[:-1][ok])]]) if len(ok) else 0.5
print('thresholds:', THR)

In [ ]:
# 9. 테스트 평가 — 깨끗한 원본 vs '차 안' 시뮬레이션
#    창 단위 지표 + 클립 단위(스무딩: 연속 프레임 중 2개 이상 초과 시 감지; 1프레임 클립은 그 1개)
def clip_detect(p, thr, n=3, k=2):   # inference_reference.py 의 smooth_n/smooth_k 와 같은 규칙
    hit = (p >= thr).astype(int)
    if len(hit) < n:
        return hit.sum() >= min(k, len(hit))
    return bool((np.convolve(hit, np.ones(n, int), 'valid') >= k).any())

metrics = {'thresholds': THR}
for split in ['test_clean', 'test_car']:
    d = D[split]; p = head.predict(d['X'], verbose=0)
    m = {}
    for i, c in enumerate(LABELS):
        yt, yp = d['Y'][:, i], (p[:, i] >= THR[c]).astype(int)
        df = pd.DataFrame({'k': d['K'], 'cat': d['C'], 'y': yt, 'p': p[:, i]})
        clip = df.groupby('k', sort=False).agg(y=('y', 'max'), det=('p', lambda s: clip_detect(s.values, THR[c])))
        neg_clips = clip[clip.y == 0]
        m[c] = dict(
            pr_auc=round(float(average_precision_score(yt, p[:, i])), 3),
            window_precision=round(float(precision_score(yt, yp, zero_division=0)), 3),
            window_recall=round(float(recall_score(yt, yp, zero_division=0)), 3),
            clip_recall=round(float(clip[clip.y == 1].det.mean()), 3),
            clip_false_alarm_rate=round(float(neg_clips.det.mean()), 4),
        )
        fp = df[(df.y == 0) & (df.p >= THR[c])].cat.value_counts().head(5)
        print(f'[{split}] {c}: {m[c]}\n   오탐 상위 카테고리: {dict(fp)}')
    metrics[split] = m

# 대략적인 '시간당 오경보' (배경 클립 총 길이 기준, 사이렌+경적 합산)
neg = clips[(clips.split == 'test') & (clips.siren + clips.horn == 0)]
neg_hours = sum(sf.info(p).duration for p in neg.path) / 3600
for split in ['test_clean', 'test_car']:
    fa = sum(metrics[split][c]['clip_false_alarm_rate'] * len(neg) for c in LABELS)
    metrics[split]['false_alarms_per_hour'] = round(fa / neg_hours, 1)
    print(f'{split}: 시간당 오경보 ≈ {metrics[split]["false_alarms_per_hour"]}  (배경 {neg_hours:.2f}시간 기준, 하드 네거티브가 많아 실제 도로보다 비관적)')

In [ ]:
# 10. YAMNet + 헤드 → 하나의 TFLite (입력 float32[15600] → 출력 float32[2])
class Exporter(tf.Module):
    def __init__(self):
        super().__init__(); self.yam = yam; self.head = head
    @tf.function(input_signature=[tf.TensorSpec([WIN], tf.float32)])
    def __call__(self, waveform):
        _, emb, _ = self.yam(waveform)
        return tf.reduce_mean(self.head(emb, training=False), axis=0)

exp = Exporter()
conv = tf.lite.TFLiteConverter.from_concrete_functions([exp.__call__.get_concrete_function()], exp)
tflite_bytes = conv.convert()
OUT = '/content/out'; os.makedirs(f'{OUT}/test_audio', exist_ok=True)
open(f'{OUT}/woollim_sound.tflite', 'wb').write(tflite_bytes)
print(f'TFLite {len(tflite_bytes) / 1e6:.1f} MB')

# TF ↔ TFLite 일치 + 속도 확인
it = tf.lite.Interpreter(model_content=tflite_bytes); it.allocate_tensors()
iin, iout = it.get_input_details()[0]['index'], it.get_output_details()[0]['index']
def tfl(w):
    it.set_tensor(iin, w.astype(np.float32)); it.invoke(); return it.get_tensor(iout).reshape(-1)
import time
w = (np.random.default_rng(0).standard_normal(WIN) * 0.1).astype(np.float32)
print('TF vs TFLite 최대 차이:', float(np.abs(exp(tf.constant(w)).numpy() - tfl(w)).max()))
t0 = time.perf_counter(); [tfl(w) for _ in range(20)]
print(f'Colab CPU 창당 {(time.perf_counter() - t0) / 20 * 1000:.1f} ms (Pi 4는 대략 3~5배 느림)')

In [ ]:
# 11. 팀원 전달 파일: labels / config / metrics / 테스트 오디오 + 기대 출력(창별 확률)
def windows(y):
    if len(y) < WIN: y = np.pad(y, (0, WIN - len(y)))
    n = 1 + int(np.ceil((len(y) - WIN) / HOP))
    y = np.pad(y, (0, (n - 1) * HOP + WIN - len(y)))
    return [y[i * HOP: i * HOP + WIN] for i in range(n)]

open(f'{OUT}/labels.txt', 'w').write('\n'.join(LABELS) + '\n')
json.dump({'sample_rate': SR, 'window': WIN, 'hop': HOP, 'labels': LABELS, 'thresholds': THR,
           'input': 'float32 mono -1..1', 'output': 'sigmoid probabilities',
           'trained_on': ['ESC-50'] + (['UrbanSound8K'] if USE_US8K else []),
           'created': pd.Timestamp.now().isoformat(timespec='seconds')},
          open(f'{OUT}/model_config.json', 'w'), indent=2)
json.dump(metrics, open(f'{OUT}/metrics.json', 'w'), indent=2, ensure_ascii=False)

test = clips[clips.split == 'test']
picks = pd.concat([test[test.siren == 1].sample(3, random_state=SEED),
                   test[test.horn == 1].sample(3, random_state=SEED),
                   test[test.siren + test.horn == 0].sample(3, random_state=SEED)])
expected = {}
for k, row in enumerate(picks.itertuples()):
    kind = 'siren' if row.siren else 'horn' if row.horn else 'background'
    name = f'{kind}_{k % 3 + 1:02d}.wav'
    y = load(row.path)
    sf.write(f'{OUT}/test_audio/{name}', y, SR, subtype='FLOAT')   # float 저장 → Pi에서 비트 단위 동일 입력
    expected[name] = [tfl(w).round(5).tolist() for w in windows(y)]
json.dump(expected, open(f'{OUT}/test_audio/expected.json', 'w'), indent=1)
print(sorted(os.listdir(f'{OUT}/test_audio')))

In [ ]:
# 12-1. 발표용 시각 자료 → /content/out/figures (그림 안 라벨은 한글 폰트 문제를 피하려고 영문)
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import confusion_matrix, precision_recall_curve, average_precision_score
from sklearn.manifold import TSNE

FIG = f'{OUT}/figures'; os.makedirs(FIG, exist_ok=True)
PINK, DEEP, AMBER, GRAY = '#EC6A84', '#D94F6B', '#FFA928', '#8E8E93'
plt.rcParams.update({'font.size': 13, 'axes.spines.top': False, 'axes.spines.right': False})
P = {k: head.predict(D[k]['X'], verbose=0) for k in ['test_clean', 'test_car']}

# (1) 학습 곡선
h = hist.history
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, key, title in [(ax[0], 'loss', 'BCE loss'), (ax[1], 'pr_auc', 'PR-AUC')]:
    a.plot(h[key], color=GRAY, label='train'); a.plot(h['val_' + key], color=PINK, lw=2.5, label='val')
    a.set_title(title); a.set_xlabel('epoch'); a.legend()
fig.tight_layout(); fig.savefig(f'{FIG}/training_curve.png', dpi=200); plt.close(fig)

# (2) PR 곡선 — 원본 vs 차량 증강
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for i, c in enumerate(LABELS):
    for k, col, ls in [('test_clean', PINK, '-'), ('test_car', DEEP, '--')]:
        y, p = D[k]['Y'][:, i], P[k][:, i]
        pr, rc, _ = precision_recall_curve(y, p)
        ax[i].plot(rc, pr, color=col, ls=ls, lw=2.5, label=f"{k[5:]} (AP={average_precision_score(y, p):.3f})")
    ax[i].set_title(c); ax[i].set_xlabel('Recall'); ax[i].set_ylabel('Precision')
    ax[i].set_ylim(0, 1.02); ax[i].legend(loc='lower left')
fig.tight_layout(); fig.savefig(f'{FIG}/pr_curves.png', dpi=200); plt.close(fig)

# (3) 혼동행렬 3클래스 (배경 / 사이렌 / 경적), 임계값 적용, 행 정규화
NAMES = ['background', 'siren', 'horn']
cmap = LinearSegmentedColormap.from_list('pink', ['#FFFFFF', PINK])
to3 = lambda Y: np.where(Y[:, 0] == 1, 1, np.where(Y[:, 1] == 1, 2, 0))
def pred3(p):
    s, hh = p[:, 0] / THR['siren'], p[:, 1] / THR['horn']
    return np.where((s >= 1) & (s >= hh), 1, np.where(hh >= 1, 2, 0))
fig, ax = plt.subplots(1, 2, figsize=(11, 4.8))
for a, k in zip(ax, ['test_clean', 'test_car']):
    cm = confusion_matrix(to3(D[k]['Y']), pred3(P[k]), labels=[0, 1, 2]); cmn = cm / cm.sum(1, keepdims=True)
    a.imshow(cmn, cmap=cmap, vmin=0, vmax=1)
    for r in range(3):
        for q in range(3):
            a.text(q, r, f'{cmn[r, q] * 100:.1f}%\n({cm[r, q]})', ha='center', va='center',
                   color='white' if cmn[r, q] > 0.6 else '#262626', fontsize=12)
    a.set_xticks(range(3), NAMES); a.set_yticks(range(3), NAMES)
    a.set_xlabel('Predicted'); a.set_ylabel('True'); a.set_title(k)
    for sp in a.spines.values(): sp.set_visible(False)
fig.tight_layout(); fig.savefig(f'{FIG}/confusion_matrix.png', dpi=200); plt.close(fig)

# (4) YAMNet 임베딩 t-SNE — 사전학습 표현만으로도 클래스가 갈라지는지
d = D['test_clean']; lab = to3(d['Y']); rs = np.random.default_rng(SEED)
idx = np.concatenate([rs.choice(np.where(lab == 0)[0], min(1500, (lab == 0).sum()), replace=False),
                      np.where(lab == 1)[0], np.where(lab == 2)[0]])
Z = TSNE(2, perplexity=30, init='pca', random_state=SEED).fit_transform(d['X'][idx])
fig, a = plt.subplots(figsize=(7, 6))
for v, name, col, s, al in [(0, 'background', '#C7C7CC', 8, 0.35), (1, 'siren', PINK, 14, 0.85), (2, 'horn', AMBER, 14, 0.85)]:
    m = lab[idx] == v; a.scatter(Z[m, 0], Z[m, 1], s=s, c=col, alpha=al, label=name, edgecolors='none')
a.set_xticks([]); a.set_yticks([]); a.set_title('YAMNet embeddings (t-SNE, test set)'); a.legend(markerscale=2)
for sp in a.spines.values(): sp.set_visible(False)
fig.tight_layout(); fig.savefig(f'{FIG}/embedding_tsne.png', dpi=200); plt.close(fig)

# (5) 시간축 예시 — 스펙트로그램 + 창별 확률 (test_audio 와 같은 클립)
def timeline(path, name):
    y = load(path); ws = windows(y); pr = np.array([tfl(w) for w in ws])
    t = np.arange(len(ws)) * HOP / SR + WIN / SR / 2
    M = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_fft=400, hop_length=160, n_mels=64), ref=np.max)
    fig, ax = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, gridspec_kw={'height_ratios': [1.2, 1]})
    ax[0].imshow(M, origin='lower', aspect='auto', cmap='magma', extent=[0, len(y) / SR, 0, 64])
    ax[0].set_ylabel('mel bin'); ax[0].set_title(f'{name}: log-mel spectrogram')
    for i, c, col in [(0, 'siren', PINK), (1, 'horn', AMBER)]:
        ax[1].plot(t, pr[:, i], marker='o', color=col, lw=2.5, label=c)
        ax[1].axhline(THR[c], color=col, ls=':', lw=1.2)
    ax[1].set_ylim(0, 1.02); ax[1].set_xlabel('time (s)'); ax[1].set_ylabel('probability'); ax[1].legend(loc='upper right')
    fig.tight_layout(); fig.savefig(f'{FIG}/timeline_{name}.png', dpi=200); plt.close(fig)
for k, name in [(0, 'siren'), (3, 'horn'), (6, 'background')]:
    timeline(picks.iloc[k].path, name)

print(sorted(os.listdir(FIG)))

In [ ]:
# 12. 다운로드 → 저장소 ml/ 폴더에 압축 풀기
!cd /content/out && zip -qr /content/woollim_model.zip .
from google.colab import files
files.download('/content/woollim_model.zip')